<a href="https://colab.research.google.com/github/di-yeferson/analitica-empresarial-integrada/blob/main/LABORATORIO_C3_AEI_DIEGO_YEFERSON_2026_02.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ANALÍTICA EMPRESARIAL INTEGRADA
## LABORATORIO CALIFICADO N.° 03
### Experimentación y decisión bajo incertidumbre

Evaluación de una campaña comercial mediante A/B testing y simulación con datos reales

| Dato | Detalle |
|---|---|
| Curso | Analítica Empresarial Integrada  |
| Semanas  | 5 y 6 --- Periodo 2026-II |
| Docente | Pilar Rocío Sayán Mejía |
| Estudiante | Quispe Huamani, Diego Yeferson |
| Sección | 6C28  |
| Fecha | 27/09/2026|
| Entorno | Google Colab · Python |
| Fuente | Hillstrom — MineThatData, experimento real publicado en 2008 |

---


### Pregunta de negocio
**¿Conviene ampliar la campaña, con qué evidencia experimental y económica se sostiene la decisión y qué riesgo está dispuesta a aceptar la gerencia?**

## Identificación del equipo

 EQUIPO N.° 5
---
| N.° | Apellidos y nombres | Código | Aporte principal |
|---|---|---|---|
| 1 | Quispe Huamani, Diego Yeferson | | Carga, trazabilidad y calidad ; diseño y muestra ; análisis A/B 😎 |
| 2 |Joel Ronald Nino de Guzman Rodriguez | | |
| 3 | Renzo Espinoza Tuesta | | |
---

Cada integrante puede explicar el código y los resultados presentados.

## Ruta de trabajo

1. Bloque 0 — Carga suministrada, entorno y parámetros del caso.
2. Bloque 1 — Trazabilidad y calidad de datos.
3. Bloque 2 — Diseño experimental y muestra.
4. Bloque 3 — Evidencia del experimento A/B.
5. Bloque 4 — Incertidumbre del beneficio esperado (bootstrap).
6. Bloque 5 — Monte Carlo y riesgo futuro.
7. Bloque 6 — Motor de decisión y sensibilidad.
8. Preguntas escritas, informe de recomendación y declaración de uso de IA.

> Los parámetros, la comparación A/B, el MDE, los umbrales y la regla de decisión se fijan antes de observar los resultados comparativos y no se modifican después.



## BLOQUE 0 — Carga, entorno y parámetros
### Celda 1 — Carga inicial de los datos (bloque suministrado por la docente)
Copia el BLoque, sin cambios.


In [1]:
import io
import hashlib
from datetime import datetime, timezone
from pathlib import Path
import pandas as pd
import requests

base = "http://www.minethatdata.com/"
archivo = (
    "Kevin_Hillstrom_MineThatData_"
    "E-MailAnalytics_DataMiningChallenge_"
    "2008.03.20.csv"
)
url = base + archivo
respuesta = requests.get(url, timeout=60)
respuesta.raise_for_status()
datos = respuesta.content
Path("hillstrom_original.csv").write_bytes(datos)
df = pd.read_csv(io.BytesIO(datos))
print("Fuente:", url)
print("Fecha UTC:", datetime.now(timezone.utc))
print("SHA-256:", hashlib.sha256(datos).hexdigest())
print("Dimensiones:", df.shape)
print("Columnas:", df.columns.tolist())
display(df.head())

Fuente: http://www.minethatdata.com/Kevin_Hillstrom_MineThatData_E-MailAnalytics_DataMiningChallenge_2008.03.20.csv
Fecha UTC: 2026-09-27 15:37:17.296538+00:00
SHA-256: 0e5893329d8b93cefecc571777672028290ab69865718020c78c7284f291aece
Dimensiones: (64000, 12)
Columnas: ['recency', 'history_segment', 'history', 'mens', 'womens', 'zip_code', 'newbie', 'channel', 'segment', 'visit', 'conversion', 'spend']


,recency,history_segment,history,mens,womens,zip_code,newbie,channel,segment,visit,conversion,spend
0,10,2) $100 - $200,142.44,1,0,Surburban,0,Phone,Womens E-Mail,0,0,0.0
1,6,3) $200 - $350,329.08,1,1,Rural,1,Web,No E-Mail,0,0,0.0
2,7,2) $100 - $200,180.65,0,1,Surburban,1,Web,Womens E-Mail,0,0,0.0
3,9,5) $500 - $750,675.83,1,0,Rural,1,Web,Mens E-Mail,0,0,0.0
4,2,1) $0 - $100,45.34,1,0,Urban,0,Web,Womens E-Mail,0,0,0.0


### Celda 2 — Entorno de trabajo y parámetros académicos del caso
Usamos las mismas librerías de los laboratorios D5 y D6. Todos los parámetros del caso se escriben aquí, en mayúsculas y antes de mirar cualquier resultado, para que funcionen como nuestro preregistro: si alguien quisiera cambiar un umbral después, tendría que hacerlo a la vista.

In [2]:
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display
from scipy.stats import norm
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 150)

# Huella oficial de la guía: con ella comprobamos que el archivo es el original.
HUELLA_REFERENCIA = "0e5893329d8b93cefecc571777672028290ab69865718020c78c7284f291aece"

# Comparación fijada por la guía antes de ver resultados.
GRUPO_A = "No E-Mail"
GRUPO_B = "Mens E-Mail"
GRUPO_EXCLUIDO = "Womens E-Mail"

# Parámetros estadísticos (contraste bilateral).
ALPHA = 0.05
POTENCIA = 0.80
MDE_BASE = 0.005
MDES = [0.003, 0.005, 0.008]

# Supuestos económicos de la gerencia: no vienen del archivo.
N_FUTURO = 10_000
MARGEN = 0.40
COSTO_BASE = 0.05
COSTOS = [0.02, 0.05, 0.10]

# Simulación reproducible y apetito de riesgo.
REPLICAS = 5_000
SEMILLA = 2026
TOLERANCIA_BASE = 0.10
TOLERANCIAS = [0.05, 0.10, 0.20]

parametros = pd.DataFrame({
    "parametro": ["Control A", "Tratamiento B", "Contraste", "Alfa", "Potencia",
                  "MDE base (pp)", "MDE sensibilidad (pp)", "Escala futura N",
                  "Margen sobre ingresos", "Costo por destinatario (USD)",
                  "Costos de sensibilidad (USD)", "Réplicas / semilla",
                  "Tolerancia base", "Tolerancias de sensibilidad"],
    "valor": [GRUPO_A, GRUPO_B, "Bilateral", ALPHA, POTENCIA, MDE_BASE * 100,
              [m * 100 for m in MDES], N_FUTURO, MARGEN, COSTO_BASE, COSTOS,
              f"{REPLICAS} / {SEMILLA}", TOLERANCIA_BASE, TOLERANCIAS]
})
display(parametros)

,parametro,valor
0,Control A,No E-Mail
1,Tratamiento B,Mens E-Mail
2,Contraste,Bilateral
3,Alfa,0.05
4,Potencia,0.8
5,MDE base (pp),0.5
6,MDE sensibilidad (pp),"[0.3, 0.5, 0.8]"
7,Escala futura N,10000
8,Margen sobre ingresos,0.4
9,Costo por destinatario (USD),0.05


---

**SE DESARROLLA EN 6 BLOQUES** 👀

---

---
## Bloque 1 — Trazabilidad y calidad de datos

### Ejercicio 1 — Trazabilidad y calidad de datos

Antes de calcular cualquier cifra comprobamos que el archivo sea exactamente el que publicó la fuente y medimos su calidad **antes** de decidir qué hacer con ella. Un cliente que no compró tiene gasto cero y es un registro válido: forma parte del denominador del experimento.

In [3]:
# Recalculamos la huella sobre los bytes descargados y la comparamos con la de
# la guía. Si no coincide, el notebook se detiene: ninguna cifra sería defendible.
huella = hashlib.sha256(datos).hexdigest()
fecha_consulta = datetime.now(timezone.utc)

trazabilidad = pd.DataFrame({
    "elemento": ["Fuente", "Fecha de consulta (UTC)", "Filas", "Columnas",
                 "SHA-256 obtenido", "SHA-256 de referencia", "¿Coincide la huella?"],
    "valor": [url, fecha_consulta.strftime("%Y-%m-%d %H:%M:%S"), df.shape[0], df.shape[1],
              huella, HUELLA_REFERENCIA, huella == HUELLA_REFERENCIA]
})
display(trazabilidad)
assert huella == HUELLA_REFERENCIA, "La fuente descargada no coincide con la publicada."

,elemento,valor
0,Fuente,http://www.minethatdata.com/Kevin_Hillstrom_Mi...
1,Fecha de consulta (UTC),2026-09-27 15:39:51
2,Filas,64000
3,Columnas,12
4,SHA-256 obtenido,0e5893329d8b93cefecc571777672028290ab698657180...
5,SHA-256 de referencia,0e5893329d8b93cefecc571777672028290ab698657180...
6,¿Coincide la huella?,True


In [4]:
# Clientes por grupo asignado, para ver si el reparto fue equilibrado.
clientes_grupo = df["segment"].value_counts().rename_axis("segment").to_frame("clientes")
clientes_grupo["porcentaje"] = (clientes_grupo["clientes"] / len(df) * 100).round(2)
display(clientes_grupo)

# El universo A/B se arma con .copy() para no tocar df. Womens E-Mail sale solo
# por alcance, y el assert confirma que no se perdió ningún registro en el filtro.
ab = df[df["segment"].isin([GRUPO_A, GRUPO_B])].copy()
excluidos = int((df["segment"] == GRUPO_EXCLUIDO).sum())
assert len(ab) + excluidos == len(df)
print("Registros que quedan en A/B:", len(ab))
print("Registros excluidos por alcance (Womens E-Mail):", excluidos)

,clientes,porcentaje
segment,,
Womens E-Mail,21387,33.42
Mens E-Mail,21307,33.29
No E-Mail,21306,33.29


Registros que quedan en A/B: 42613
Registros excluidos por alcance (Womens E-Mail): 21387


**Pregunta 1.1.** ¿La huella coincide con la referencia? Registre fuente, fecha, dimensiones y clientes por grupo. ¿Cuántos registros quedan en A/B y cuántos se excluyen por alcance?

**Respuesta.** Sí, la huella SHA-256 obtenida (`0e5893…aece`) coincide exactamente con la referencia, así que el archivo es el original. Fuente: minethatdata.com (Hillstrom, 2008), consultada el 27/09/2026 (UTC), con 64 000 filas y 12 columnas. Por grupo hay 21 306 clientes en No E-Mail, 21 307 en Mens E-Mail y 21 387 en Womens E-Mail. En A/B quedan **42 613 registros** y se excluyen **21 387** (Womens E-Mail) solo por alcance, no por calidad.

### Inventario de calidad antes de decidir

Aplicamos los mismos controles a la base completa y al universo A/B. Primero se cuantifica cada hallazgo y recién después se decide su tratamiento.

In [5]:
indicadores = ["conversion", "visit", "mens", "womens", "newbie"]

# Cada control es una condición distinta; los no compradores con gasto 0 se
# cuentan aparte como válidos para no confundirlos con errores.
def controles_calidad(datos_eval):
    return {
        "Valores ausentes": int(datos_eval.isna().sum().sum()),
        "Indicadores fuera de 0/1": int(sum((~datos_eval[c].isin([0, 1])).sum() for c in indicadores)),
        "Gastos negativos": int((datos_eval["spend"] < 0).sum()),
        "Compra con gasto 0": int(((datos_eval["conversion"] == 1) & (datos_eval["spend"] == 0)).sum()),
        "Sin compra con gasto > 0": int(((datos_eval["conversion"] == 0) & (datos_eval["spend"] > 0)).sum()),
        "Compra sin visita": int(((datos_eval["conversion"] == 1) & (datos_eval["visit"] == 0)).sum()),
        "Filas idénticas (repeticiones)": int(datos_eval.duplicated().sum()),
        "No compradores con gasto 0 (válidos)": int(((datos_eval["conversion"] == 0) & (datos_eval["spend"] == 0)).sum()),
    }

calidad = pd.DataFrame({"base_completa": controles_calidad(df),
                        "universo_A_B": controles_calidad(ab)})
display(calidad)

# Las filas repetidas se miden pero no se borran: sin identificador de cliente,
# dos filas iguales pueden ser dos clientes distintos.
print("Filas A/B que comparten perfil idéntico con otra fila:", len(ab[ab.duplicated(keep=False)]))
print("Registros eliminados por calidad:", 0)
print("Universo analizado A/B:", len(ab))

,base_completa,universo_A_B
Valores ausentes,0,0
Indicadores fuera de 0/1,0,0
Gastos negativos,0,0
Compra con gasto 0,0,0
Sin compra con gasto > 0,0,0
Compra sin visita,0,0
Filas idénticas (repeticiones),6562,4349
No compradores con gasto 0 (válidos),63422,42224


Filas A/B que comparten perfil idéntico con otra fila: 5059
Registros eliminados por calidad: 0
Universo analizado A/B: 42613


**Pregunta 1.2.** ¿Hay ausentes, indicadores fuera de 0/1, gastos negativos o inconsistencias entre conversion y spend? Cuantifique los hallazgos antes de decidir y documente su tratamiento. ¿Por qué no debe eliminar filas idénticas automáticamente?

**Respuesta.** No hay ausentes, indicadores fuera de 0/1, gastos negativos ni incoherencias entre `conversion` y `spend` (0 compras con gasto 0 y 0 gastos sin compra); tampoco compras sin visita. Los 42 224 clientes A/B que no compraron tienen gasto 0 y son **válidos**: forman parte del denominador. Hay 4 349 filas repetidas en A/B, pero no se eliminan: el archivo no tiene identificador y con 12 variables de pocos valores dos clientes distintos pueden tener el mismo perfil (por ejemplo, sin compra y con igual history). Borrarlas reduciría el universo asignado sin prueba de duplicación. **Tratamiento: 0 registros eliminados.**

### Balance basal entre A y B

Comparamos recency e history, dos variables medidas antes del experimento. Usamos la diferencia estandarizada porque permite comparar variables con escalas distintas (meses y dólares).

In [6]:
# Diferencia estandarizada = (media B − media A) / desviación combinada.
# Por convención, valores menores a 0.1 indican buen balance.
def diferencia_estandarizada(columna):
    a = ab.loc[ab["segment"] == GRUPO_A, columna]
    b = ab.loc[ab["segment"] == GRUPO_B, columna]
    return (b.mean() - a.mean()) / np.sqrt((a.var(ddof=1) + b.var(ddof=1)) / 2)

# Incluimos la mediana porque history es asimétrica.
basal = ab.groupby("segment")[["recency", "history"]].agg(["mean", "median", "std"]).round(3)
display(basal.loc[[GRUPO_A, GRUPO_B]])

balance = pd.DataFrame({
    "variable": ["recency", "history"],
    "media_A": [ab.loc[ab["segment"] == GRUPO_A, c].mean() for c in ["recency", "history"]],
    "media_B": [ab.loc[ab["segment"] == GRUPO_B, c].mean() for c in ["recency", "history"]],
    "diferencia_estandarizada": [diferencia_estandarizada(c) for c in ["recency", "history"]]
}).round(4)
balance["diferencia_B_A"] = (balance["media_B"] - balance["media_A"]).round(4)
display(balance)

recency                history                  
               mean median    std     mean   median      std
segment                                                     
No E-Mail     5.750    5.0  3.498  240.883  156.655  252.739
Mens E-Mail   5.774    6.0  3.513  242.836  157.220  260.356

,variable,media_A,media_B,diferencia_estandarizada,diferencia_B_A
0,recency,5.7497,5.7736,0.0068,0.0239
1,history,240.8827,242.8359,0.0076,1.9532


In [7]:
# Distribución de recency en porcentaje, para que el tamaño de cada grupo no
# influya en la comparación visual.
fig = px.histogram(ab, x="recency", color="segment", barmode="group", histnorm="percent",
                   category_orders={"segment": [GRUPO_A, GRUPO_B]},
                   title="Distribución basal de recency por grupo")
fig.update_xaxes(title="Meses desde la última compra", dtick=1)
fig.update_yaxes(title="Clientes (%)")
fig.show()

**Pregunta 1.3.** Compare recency e history entre A y B. ¿Observa diferencias basales? Explique por qué esta revisión no prueba ni refuta por sí sola la asignación aleatoria descrita por la fuente.

**Respuesta.** No se observan diferencias basales relevantes: recency promedia 5.750 meses en A y 5.774 en B, y history USD 240.88 frente a USD 242.84. Las diferencias estandarizadas son 0.007 y 0.008, muy por debajo de 0.1. Aun así, esta revisión no prueba la aleatorización: solo mira dos variables observadas, y un buen balance también puede darse sin sorteo. Tampoco la refutaría una diferencia aislada, porque un sorteo correcto puede producir desbalances por azar. La garantía viene del mecanismo de asignación que describe la fuente.

---
## Bloque 2 — Diseño experimental y muestra

### Ejercicio 2 — Diseño experimental y muestra

Antes de comparar resultados dejamos escrito el diseño: qué se compara, sobre quién, con qué hipótesis y con qué exigencia. Después calculamos cuántos clientes necesitaría un nuevo experimento usando **solo** la tasa del control, sin mirar el efecto observado.

In [9]:
# Ficha del experimento registrada antes de calcular resultados comparativos.
diseno = pd.DataFrame({
    "elemento": ["Control A", "Tratamiento B", "Unidad experimental", "Hipótesis nula",
                 "Hipótesis alternativa", "Métrica primaria", "MDE", "Alfa", "Potencia",
                 "Universo"],
    "definicion": [GRUPO_A, GRUPO_B, "Cliente", "H0: pB = pA", "H1: pB ≠ pA (bilateral)",
                   "conversion = compradores / clientes asignados",
                   f"{MDE_BASE*100:.1f} pp (sensibilidad 0.3 y 0.8 pp)", ALPHA, POTENCIA,
                   "Todos los clientes asignados a A y B"]
})
display(diseno)

,elemento,definicion
0,Control A,No E-Mail
1,Tratamiento B,Mens E-Mail
2,Unidad experimental,Cliente
3,Hipótesis nula,H0: pB = pA
4,Hipótesis alternativa,H1: pB ≠ pA (bilateral)
5,Métrica primaria,conversion = compradores / clientes asignados
6,MDE,0.5 pp (sensibilidad 0.3 y 0.8 pp)
7,Alfa,0.05
8,Potencia,0.8
9,Universo,Todos los clientes asignados a A y B


In [10]:
# Guardrails propuestos para un NUEVO experimento. Hillstrom no registra bajas
# ni quejas, así que aquí solo se declaran umbral, acción y dato faltante.
guardrails = pd.DataFrame({
    "guardrail": ["Tasa de baja de suscripción", "Tasa de quejas por correo no deseado"],
    "metrica": ["Clientes que se dan de baja / clientes asignados",
                "Clientes que marcan el correo como spam / clientes asignados"],
    "umbral": ["Límite superior unilateral 95 % de B − A < +0.20 pp",
               "Límite superior unilateral 95 % de B − A < +0.05 pp"],
    "accion": ["Si se supera: no ampliar y revisar frecuencia y contenido del correo",
               "Si se supera: detener envíos y revisar segmentación y remitente"],
    "dato_faltante": ["Registro de bajas por cliente y fecha",
                      "Registro de quejas por cliente y fecha"]
})
display(guardrails)
print("Los guardrails son una propuesta para un nuevo experimento; no son evidencia observada.")

,guardrail,metrica,umbral,accion,dato_faltante
0,Tasa de baja de suscripción,Clientes que se dan de baja / clientes asignados,Límite superior unilateral 95 % de B − A < +0....,Si se supera: no ampliar y revisar frecuencia ...,Registro de bajas por cliente y fecha
1,Tasa de quejas por correo no deseado,Clientes que marcan el correo como spam / clie...,Límite superior unilateral 95 % de B − A < +0....,Si se supera: detener envíos y revisar segment...,Registro de quejas por cliente y fecha


Los guardrails son una propuesta para un nuevo experimento; no son evidencia observada.


**Pregunta 2.1.** Antes de calcular resultados comparativos, registre A, B, unidad, hipótesis, métrica, MDE, α y potencia. Proponga dos guardrails para un nuevo experimento, con umbral y acción; indique qué datos faltan para evaluarlos.

**Respuesta.** El diseño quedó registrado antes de comparar resultados: A = No E-Mail, B = Mens E-Mail, unidad = cliente, H₀: pB = pA frente a H₁: pB ≠ pA, métrica = conversion, MDE = 0.5 pp, α = 0.05 y potencia = 0.80. Para un nuevo experimento se proponen dos guardrails: **tasa de bajas de suscripción** (límite superior de B − A < +0.20 pp; si se supera, no ampliar) y **tasa de quejas por spam** (límite < +0.05 pp; si se supera, detener envíos). Hillstrom no registra bajas ni quejas, así que faltan esos datos por cliente para evaluarlos.

### Tamaño de muestra con el baseline del control

Usamos `NormalIndPower` y `proportion_effectsize` como en el Lab D6, pero con `alternative="two-sided"` porque el contraste del caso es bilateral.

In [11]:
# Baseline = tasa del control. No se usa la tasa de B ni el efecto observado.
baseline = ab.loc[ab["segment"] == GRUPO_A, "conversion"].mean()
n_A_real = int((ab["segment"] == GRUPO_A).sum())
n_B_real = int((ab["segment"] == GRUPO_B).sum())
analysis = NormalIndPower()

# Para cada MDE: tamaño del efecto (h de Cohen) y clientes por brazo.
# np.ceil redondea hacia arriba porque no existen fracciones de cliente.
filas = []
for mde in MDES:
    es = proportion_effectsize(baseline + mde, baseline)
    n = analysis.solve_power(effect_size=es, alpha=ALPHA, power=POTENCIA,
                             ratio=1, alternative="two-sided")
    filas.append({"MDE_pp": mde * 100,
                  "tasa_B_planificada": baseline + mde,
                  "n_por_brazo": int(np.ceil(n)),
                  "n_total": 2 * int(np.ceil(n))})
muestra = pd.DataFrame(filas)
muestra["n_real_A"] = n_A_real
muestra["n_real_B"] = n_B_real
muestra["tamanos_reales_cubren"] = muestra["n_por_brazo"] <= min(n_A_real, n_B_real)
print(f"Baseline (tasa del control): {baseline:.4%}")
display(muestra)

Baseline (tasa del control): 0.5726%


,MDE_pp,tasa_B_planificada,n_por_brazo,n_total,n_real_A,n_real_B,tamanos_reales_cubren
0,0.3,0.008726,12377,24754,21306,21307,True
1,0.5,0.010726,5003,10006,21306,21307,True
2,0.8,0.013726,2259,4518,21306,21307,True


In [12]:
# Cálculo inverso: qué MDE se podía detectar con los tamaños reales,
# de nuevo usando solo el baseline.
h_detectable = analysis.solve_power(nobs1=n_A_real, alpha=ALPHA, power=POTENCIA,
                                    ratio=n_B_real / n_A_real, alternative="two-sided")
tasa_detectable = np.sin(np.arcsin(np.sqrt(baseline)) + h_detectable / 2) ** 2
print(f"MDE alcanzable con los tamaños reales: {(tasa_detectable - baseline)*100:.3f} pp")

fig = px.line(muestra, x="MDE_pp", y="n_por_brazo", markers=True,
              title="Clientes por brazo según el MDE (contraste bilateral)",
              labels={"MDE_pp": "MDE (pp)", "n_por_brazo": "Clientes por brazo"})
fig.add_hline(y=min(n_A_real, n_B_real), line_dash="dash", line_color="red",
              annotation_text="Tamaño real por brazo")
fig.show()

MDE alcanzable con los tamaños reales: 0.223 pp


**Pregunta 2.2.** Usando solo la tasa del control como baseline, ¿cuántos clientes por brazo requiere un nuevo experimento 1:1 para detectar cada MDE establecido con contraste bilateral? Compare con los tamaños reales de A y B.

**Respuesta.** Con la tasa del control como baseline (0.5726 %) y contraste bilateral, un nuevo experimento 1:1 necesita **12 377** clientes por brazo para 0.3 pp, **5 003** para 0.5 pp y **2 259** para 0.8 pp. Los tamaños reales (21 306 en A y 21 307 en B) superan los tres requisitos; con ellos, el MDE alcanzable al 80 % de potencia era de 0.223 pp.

In [13]:
# Costo de exigir un MDE más fino: cuántas veces crece la muestra frente al
# MDE base, cuánto cuesta enviar el correo al brazo B y cuántos clientes del
# control se quedan sin campaña durante la prueba.
costo_mde = muestra[["MDE_pp", "n_por_brazo", "n_total"]].copy()
n_ref = costo_mde.loc[np.isclose(costo_mde["MDE_pp"], MDE_BASE * 100), "n_por_brazo"].iloc[0]
costo_mde["veces_muestra_vs_0.5pp"] = (costo_mde["n_por_brazo"] / n_ref).round(2)
costo_mde["costo_envio_B_USD"] = costo_mde["n_por_brazo"] * COSTO_BASE
costo_mde["control_sin_campana"] = costo_mde["n_por_brazo"]
display(costo_mde)

,MDE_pp,n_por_brazo,n_total,veces_muestra_vs_0.5pp,costo_envio_B_USD,control_sin_campana
0,0.3,12377,24754,2.47,618.85,12377
1,0.5,5003,10006,1.00,250.15,5003
2,0.8,2259,4518,0.45,112.95,2259


**Pregunta 2.3.** ¿Qué costo operativo implica reducir el MDE? Distinga la planificación retrospectiva de un preregistro auténtico. No use el efecto observado para calcular potencia y justificar el resultado.

**Respuesta.** Bajar el MDE de 0.5 a 0.3 pp multiplica la muestra por 2.47 (de 5 003 a 12 377 por brazo). Eso sube el costo de envío en B de USD 250.15 a USD 618.85, deja a más clientes del control sin campaña y alarga el reclutamiento. Aquí la planificación es **retrospectiva**: el experimento ya ocurrió y solo usamos el baseline del control para ver si la muestra alcanzaba. Un **preregistro auténtico** fija hipótesis, MDE y muestra antes de recoger datos. No usamos el efecto observado para calcular la potencia, porque eso solo confirmaría lo ya visto.

---
## Bloque 3 — Evidencia del experimento A/B

### Ejercicio 3 — Evidencia del experimento A/B

Comparamos la conversión de A y B con **todos** los clientes asignados. Adaptamos la función `diferencia_wald` del Lab D6 al caso bilateral: el p-valor y el IC salen del mismo error estándar, así que sus conclusiones son coherentes entre sí.

In [14]:
# Compradores, tamaño y tasa de cada brazo. Los que no compraron entran en el
# count, es decir, siguen en el denominador.
tabla = ab.groupby("segment")["conversion"].agg(["sum", "count", "mean"]).loc[[GRUPO_A, GRUPO_B]]
tabla.columns = ["compradores", "n", "tasa_conversion"]
x_A, n_A = int(tabla.loc[GRUPO_A, "compradores"]), int(tabla.loc[GRUPO_A, "n"])
x_B, n_B = int(tabla.loc[GRUPO_B, "compradores"]), int(tabla.loc[GRUPO_B, "n"])
p_A, p_B = x_A / n_A, x_B / n_B
display(tabla)

,compradores,n,tasa_conversion
segment,,,
No E-Mail,122,21306,0.005726
Mens E-Mail,267,21307,0.012531


In [15]:
def diferencia_wald(x_b, n_b, x_a, n_a, alpha=ALPHA):
    # Con menos de 10 compras o no compras la aproximación normal no es confiable.
    if min(x_b, n_b - x_b, x_a, n_a - x_a) < 10:
        raise ValueError("Frecuencias pequeñas: revisar método antes de continuar")
    pb, pa = x_b / n_b, x_a / n_a
    d = pb - pa
    se = np.sqrt(pb * (1 - pb) / n_b + pa * (1 - pa) / n_a)   # error estándar no combinado
    z = d / se
    z_crit = norm.ppf(1 - alpha / 2)                          # 1.96 para el IC bilateral
    return {"d": d, "se": se, "z": z,
            "p_bilateral": 2 * norm.sf(abs(z)),               # H1: pB ≠ pA
            "ic95_low": d - z_crit * se,
            "ic95_high": d + z_crit * se}

conv = diferencia_wald(x_B, n_B, x_A, n_A)
lift_rel = conv["d"] / p_A
print(f"Diferencia B − A: {conv['d']*100:.3f} pp")
print(f"Lift relativo: {lift_rel:.2%}")
print(f"z: {conv['z']:.4f}")
print(f"p-valor bilateral: {conv['p_bilateral']:.6g}")
print(f"IC bilateral 95 %: [{conv['ic95_low']*100:.4f}, {conv['ic95_high']*100:.4f}] pp")

Diferencia B − A: 0.681 pp
Lift relativo: 118.84%
z: 7.3899
p-valor bilateral: 1.46929e-13
IC bilateral 95 %: [0.5000, 0.8610] pp


In [16]:
# Tasas por brazo con el valor exacto sobre cada barra.
grafico = tabla.reset_index()
grafico["conversion_pct"] = grafico["tasa_conversion"] * 100
fig = px.bar(grafico, x="segment", y="conversion_pct", text="conversion_pct",
             title="Conversión observada por brazo — Hillstrom",
             labels={"conversion_pct": "Clientes que compraron (%)", "segment": "Brazo"})
fig.update_traces(texttemplate="%{text:.3f}%", textposition="outside")
fig.update_yaxes(rangemode="tozero")
fig.show()

**Pregunta 3.1.** ¿Cuáles son el número de compradores, tamaño y tasa de conversión de cada brazo? Calcule diferencia B − A en puntos porcentuales, lift relativo, IC bilateral del 95 % y p-valor bilateral para la diferencia de proporciones.

**Respuesta.** El control A tuvo **122 compradores de 21 306** (0.573 %) y B **267 de 21 307** (1.253 %). La diferencia B − A es **+0.681 pp**, con un lift relativo de **118.84 %**. El IC bilateral del 95 % va de **0.500 a 0.861 pp** y el p-valor bilateral es **1.47 × 10⁻¹³** (z = 7.39). El p-valor indica qué tan raro sería observar una diferencia así si H₀ fuera cierta; no es la probabilidad de que H₀ sea verdadera.

In [17]:
# Efecto con su IC y dos referencias: el cero (sin efecto) y el MDE.
d_pp = conv["d"] * 100
fig = go.Figure(go.Scatter(
    x=[d_pp], y=["B − A"], mode="markers",
    marker=dict(size=14, color="navy"),
    error_x=dict(type="data", symmetric=False,
                 array=[conv["ic95_high"] * 100 - d_pp],
                 arrayminus=[d_pp - conv["ic95_low"] * 100])))
fig.add_vline(x=0, line_dash="dash", line_color="gray", annotation_text="Sin efecto")
fig.add_vline(x=MDE_BASE * 100, line_dash="dash", line_color="red", annotation_text="MDE 0.5 pp")
fig.update_layout(title="Efecto en conversión (B − A) con IC bilateral 95 %",
                  xaxis_title="Diferencia en puntos porcentuales", height=320)
fig.show()

# Tres lecturas separadas para no mezclar significancia con relevancia.
lectura = pd.Series({
    "Evidencia de mejora (p < 0.05 e IC > 0)": bool(conv["p_bilateral"] < ALPHA and conv["ic95_low"] > 0),
    "Efecto observado ≥ MDE": bool(conv["d"] >= MDE_BASE),
    "Límite inferior del IC > MDE": bool(conv["ic95_low"] > MDE_BASE)
})
display(lectura.to_frame("cumple"))

,cumple
Evidencia de mejora (p < 0.05 e IC > 0),True
Efecto observado ≥ MDE,True
Límite inferior del IC > MDE,True


**Pregunta 3.2.** Grafique el efecto y su intervalo. ¿Existe evidencia de mejora y el efecto observado alcanza el MDE? Distinga mejora estadística, relevancia práctica y la evidencia necesaria para sostener que el efecto verdadero supera el MDE.

**Respuesta.** Sí hay evidencia de mejora: p < 0.05 y el IC excluye el cero. El efecto observado (0.681 pp) alcanza el MDE de 0.5 pp. Para **sostener que el efecto verdadero supera el MDE** hace falta que el límite inferior del IC esté por encima de 0.5 pp; aquí queda en 0.50002 pp, justo en el borde, así que esa evidencia es muy ajustada. La significancia estadística dice que la mejora no es azar; la relevancia práctica depende de si genera dinero, y eso se evalúa en los ejercicios 4 y 5.

In [18]:
# Ingreso medio por cliente INCLUYENDO los ceros: es la métrica económica
# correcta, porque el correo se envía (y se paga) para todos los clientes.
ingreso = ab.groupby("segment")["spend"].agg(ingreso_medio_cliente="mean", clientes="count")

# Gasto solo entre compradores: se calcula únicamente para mostrar el sesgo.
compradores = ab[ab["conversion"] == 1].groupby("segment")["spend"].agg(
    gasto_medio_comprador="mean", compradores="count")
ingreso = ingreso.join(compradores).loc[[GRUPO_A, GRUPO_B]]
display(ingreso.round(4))

dif_ingreso = ingreso.loc[GRUPO_B, "ingreso_medio_cliente"] - ingreso.loc[GRUPO_A, "ingreso_medio_cliente"]
dif_compradores = ingreso.loc[GRUPO_B, "gasto_medio_comprador"] - ingreso.loc[GRUPO_A, "gasto_medio_comprador"]
print(f"Diferencia de ingreso medio por cliente (B − A): USD {dif_ingreso:.4f}")
print(f"Diferencia de gasto medio solo entre compradores (B − A): USD {dif_compradores:.4f}")

,ingreso_medio_cliente,clientes,gasto_medio_comprador,compradores
segment,,,,
No E-Mail,0.6528,21306,114.0027,122
Mens E-Mail,1.4226,21307,113.5269,267


Diferencia de ingreso medio por cliente (B − A): USD 0.7698
Diferencia de gasto medio solo entre compradores (B − A): USD -0.4758


**Pregunta 3.3.** Calcule el ingreso medio por cliente de A y B incluyendo los ceros. ¿Por qué comparar solo compradores puede sesgar la decisión sobre la campaña?

**Respuesta.** Incluyendo los ceros, el ingreso medio por cliente es **USD 0.6528 en A** y **USD 1.4226 en B**, una diferencia de **USD 0.7698** por cliente. Si miráramos solo a los compradores, A gasta USD 114.00 y B USD 113.53 (−0.48), y parecería que la campaña no sirve. Ese análisis está sesgado porque la campaña cambia **quién** compra: el valor está en que compran más clientes, no en tickets más altos. Además, el correo se paga por destinatario, no por comprador.

---
## Bloque 4 — Incertidumbre del beneficio esperado

### Ejercicio 4 — Incertidumbre del beneficio esperado

Con el gasto real de cada brazo aplicamos bootstrap: remuestreamos **dentro** de A y **dentro** de B, con sus tamaños originales y con los ceros incluidos. Así medimos cuánto podría variar el beneficio esperado solo por trabajar con una muestra.

In [19]:
gasto_A = ab.loc[ab["segment"] == GRUPO_A, "spend"].to_numpy(dtype=float)
gasto_B = ab.loc[ab["segment"] == GRUPO_B, "spend"].to_numpy(dtype=float)
dif_observada = gasto_B.mean() - gasto_A.mean()

# 5 000 réplicas con semilla 2026. Cada brazo se remuestrea por separado;
# mezclarlos destruiría la comparación.
rng = np.random.default_rng(SEMILLA)
dif_boot = np.empty(REPLICAS)
for i in range(REPLICAS):
    media_A = rng.choice(gasto_A, size=len(gasto_A), replace=True).mean()
    media_B = rng.choice(gasto_B, size=len(gasto_B), replace=True).mean()
    dif_boot[i] = media_B - media_A

print("Tamaños remuestreados:", len(gasto_A), "(A) y", len(gasto_B), "(B)")
print("Ceros incluidos:", int((gasto_A == 0).sum()), "(A) y", int((gasto_B == 0).sum()), "(B)")
print(f"Diferencia observada de ingreso medio: USD {dif_observada:.4f}")
display(pd.Series(dif_boot).describe(percentiles=[0.025, 0.5, 0.975]).to_frame("dif_ingreso_medio_boot"))

Tamaños remuestreados: 21306 (A) y 21307 (B)
Ceros incluidos: 21184 (A) y 21040 (B)
Diferencia observada de ingreso medio: USD 0.7698


,dif_ingreso_medio_boot
count,5000.000000
mean,0.767622
std,0.144769
min,0.213437
2.5%,0.488663
50%,0.768008
97.5%,1.057977
max,1.389023


**Pregunta 4.1.** Con el gasto real de cada brazo, construya 5 000 réplicas bootstrap independientes: remuestree con reemplazo nA registros de A y nB de B, manteniendo los tamaños originales. En cada réplica estime la diferencia de ingreso medio por cliente.

**Respuesta.** Se generaron 5 000 réplicas bootstrap independientes remuestreando con reemplazo 21 306 gastos de A y 21 307 de B, con los ceros incluidos (21 184 en A y 21 040 en B). La diferencia de ingreso medio observada es USD 0.7698; la media de las réplicas es USD 0.7676 y sus percentiles 2.5 y 97.5 son USD 0.4887 y USD 1.0580.

In [20]:
# Fórmula de la guía: N × [margen × diferencia de ingreso medio − costo].
def beneficio_esperado(dif_ingreso_medio, costo=COSTO_BASE, n=N_FUTURO, margen=MARGEN):
    return n * (margen * dif_ingreso_medio - costo)

beneficio_puntual = beneficio_esperado(dif_observada)
beneficio_boot = beneficio_esperado(dif_boot)
p025, p975 = np.percentile(beneficio_boot, [2.5, 97.5])

resumen_boot = pd.DataFrame({
    "indicador": ["Estimación puntual", "Media de las réplicas", "Percentil 2.5", "Percentil 97.5"],
    "beneficio_USD": [beneficio_puntual, beneficio_boot.mean(), p025, p975]
}).round(2)
display(resumen_boot)

,indicador,beneficio_USD
0,Estimación puntual,2579.31
1,Media de las réplicas,2570.49
2,Percentil 2.5,1454.65
3,Percentil 97.5,3731.91


In [21]:
# Distribución bootstrap con el cero y los percentiles 2.5 y 97.5.
fig = px.histogram(x=beneficio_boot, nbins=70,
                   title="Bootstrap del beneficio incremental esperado (N = 10 000)")
fig.add_vline(x=0, line_dash="dash", line_color="red", annotation_text="Cero")
fig.add_vline(x=p025, line_dash="dot", line_color="gray", annotation_text="P2.5")
fig.add_vline(x=p975, line_dash="dot", line_color="gray", annotation_text="P97.5")
fig.update_xaxes(title="Beneficio incremental esperado (USD)")
fig.update_yaxes(title="Frecuencia")
fig.show()

**Pregunta 4.2.** Transforme cada diferencia en beneficio incremental esperado para N clientes: N × [margen × diferencia de ingreso medio − costo por destinatario]. Reporte estimación puntual, media de las réplicas y percentiles 2.5 y 97.5. Grafique la distribución y el cero.

**Respuesta.** Para N = 10 000 clientes, margen de 40 % y costo de USD 0.05, el beneficio incremental esperado es **USD 2 579.31** (estimación puntual). La media de las réplicas es **USD 2 570.49** y el intervalo bootstrap del 95 % va de **USD 1 454.65 a USD 3 731.91**. Todo el intervalo queda por encima de cero.

**Pregunta 4.3.** ¿Qué incertidumbre describe ese intervalo bootstrap? ¿Por qué no equivale a un intervalo predictivo del beneficio realizado por una campaña futura?

**Respuesta.** Ese intervalo describe la **incertidumbre de estimación** del beneficio **esperado**: cuánto podría variar la diferencia de ingreso medio si repitiéramos el experimento con muestras del mismo tamaño. No es un intervalo predictivo porque una campaña real de 10 000 clientes suma además el azar de quién compra y cuánto gasta en esa campaña concreta, algo que el promedio no captura. Tampoco incluye cambios de población o de época. Por eso el Monte Carlo del ejercicio 5 da un rango más amplio.

---
## Bloque 5 — Monte Carlo y riesgo futuro

### Ejercicio 5 — Monte Carlo y riesgo futuro

Ahora simulamos campañas futuras completas, como el Monte Carlo del Lab D5: en cada escenario se sortean 10 000 gastos de A y 10 000 de B desde sus distribuciones reales. Esto ya no describe un promedio, sino el resultado que podría tener una campaña concreta.

In [25]:
# Generador propio con semilla 2026. Guardamos las sumas de cada escenario para
# reutilizarlas en la sensibilidad del Bloque 6 y así aislar el cambio de política.
rng_mc = np.random.default_rng(SEMILLA)
suma_A = np.empty(REPLICAS)
suma_B = np.empty(REPLICAS)
for i in range(REPLICAS):
    suma_A[i] = rng_mc.choice(gasto_A, size=N_FUTURO, replace=True).sum()
    suma_B[i] = rng_mc.choice(gasto_B, size=N_FUTURO, replace=True).sum()

# Resultado incremental de la guía: margen × (suma B − suma A) − N × costo.
def resultado_incremental(suma_b, suma_a, costo=COSTO_BASE, n=N_FUTURO, margen=MARGEN):
    return margen * (suma_b - suma_a) - n * costo

resultado_mc = resultado_incremental(suma_B, suma_A)
print("Escenarios simulados:", len(resultado_mc))
display(pd.DataFrame({"suma_A": suma_A, "suma_B": suma_B,
                      "resultado_incremental": resultado_mc}).head())

Escenarios simulados: 5000


,suma_A,suma_B,resultado_incremental
0,5958.55,13136.50,2371.180
1,6588.48,18970.70,4452.888
2,4588.11,13292.42,2981.724
3,5699.99,14818.63,3147.456
4,6370.47,11485.67,1546.080


**Pregunta 5.1.** Construya 5 000 escenarios. En cada uno, extraiga con reemplazo N gastos de A y N de B desde sus distribuciones empíricas originales. Modele el resultado incremental como margen × (suma de B − suma de A) − N × costo por destinatario.

**Respuesta.** Se construyeron 5 000 escenarios. En cada uno se extrajeron con reemplazo 10 000 gastos de A y 10 000 de B desde sus distribuciones empíricas, y se calculó el resultado incremental: 0.40 × (suma B − suma A) − 10 000 × 0.05. Las sumas se guardaron para reutilizarlas en la sensibilidad.

In [26]:
# Riesgo modelado = proporción de escenarios con pérdida; P5 = escenario adverso.
beneficio_medio = resultado_mc.mean()
riesgo = np.mean(resultado_mc < 0)
p5, p50, p95 = np.percentile(resultado_mc, [5, 50, 95])

tabla_riesgo = pd.DataFrame({
    "Métrica": ["Beneficio medio simulado (USD)", "Proporción de escenarios negativos",
                "P5 (USD)", "Mediana (USD)", "P95 (USD)"],
    "Resultado": [beneficio_medio, riesgo, p5, p50, p95]
}).round(4)
display(tabla_riesgo)
print(f"En {riesgo:.2%} de los escenarios la campaña pierde dinero. "
      f"En el P5 el resultado sería USD {p5:,.2f}.")

,Métrica,Resultado
0,Beneficio medio simulado (USD),2565.6404
1,Proporción de escenarios negativos,0.0002
2,P5 (USD),1159.9668
3,Mediana (USD),2574.1560
4,P95 (USD),3936.9704


En 0.02% de los escenarios la campaña pierde dinero. En el P5 el resultado sería USD 1,159.97.


In [27]:
# Distribución de los escenarios con el cero y el P5.
fig = px.histogram(x=resultado_mc, nbins=80,
                   title="Distribución simulada del resultado incremental de la campaña futura")
fig.add_vline(x=0, line_dash="dash", line_color="red", annotation_text="Cero")
fig.add_vline(x=p5, line_dash="dash", line_color="orange", annotation_text="P5")
fig.update_xaxes(title="Resultado incremental (USD)")
fig.update_yaxes(title="Frecuencia")
fig.show()

**Pregunta 5.2.** Reporte beneficio medio, proporción de escenarios negativos, P5, mediana y P95. Grafique la distribución con el cero y el P5. ¿Qué aporta el P5 que no revela el promedio?

**Respuesta.** El beneficio medio simulado es **USD 2 565.64**, con mediana de USD 2 574.16, **P5 = USD 1 159.97** y P95 = USD 3 936.97. La proporción de escenarios negativos es **0.02 %** (1 de 5 000). El P5 muestra el escenario adverso plausible: aun en el 5 % de futuros más desfavorables se ganarían unos USD 1 160, pero es USD 1 406 menos que el promedio. Esa dispersión el promedio no la revela. Este riesgo es modelado, no una frecuencia futura garantizada.

**Pregunta 5.3.** Explique el supuesto de independencia entre las dos poblaciones futuras simuladas. ¿Por qué el modelo no observa los dos resultados potenciales del mismo cliente ni incorpora toda la incertidumbre de estimación del ejercicio 4?

**Respuesta.** El modelo simula las poblaciones futuras de A y B de forma independiente, como si fueran dos grupos distintos de 10 000 clientes. En la realidad un cliente recibe o no recibe el correo, y nunca observamos sus dos resultados potenciales (con y sin campaña); por eso no se puede simular la diferencia cliente por cliente. Además, el Monte Carlo toma las distribuciones empíricas como si fueran las verdaderas: no incorpora la incertidumbre de estimación del ejercicio 4, así que el riesgo real puede estar subestimado.

---
## Bloque 6 — Motor de decisión y sensibilidad

### Ejercicio 6 — Motor de decisión y sensibilidad

Programamos la regla de la guía como una función transparente, con el estilo del motor del Lab D5. La regla no se modifica: en la sensibilidad solo cambian los parámetros que pide la guía.

In [28]:
def motor_decision(conv, beneficio_medio, riesgo, mde=MDE_BASE,
                   tolerancia=TOLERANCIA_BASE, alpha=ALPHA):
    criterios = {
        "Efecto positivo con p < 0.05": bool(conv["d"] > 0 and conv["p_bilateral"] < alpha),
        "IC 95 % por encima de cero": bool(conv["ic95_low"] > 0),
        "Efecto observado ≥ MDE": bool(conv["d"] >= mde),
        "Beneficio medio simulado > 0": bool(beneficio_medio > 0),
        "Riesgo ≤ tolerancia": bool(riesgo <= tolerancia),
    }
    # Ampliar exige los cinco criterios; beneficio ≤ 0 lleva a no ampliar;
    # cualquier otro caso pide un nuevo experimento.
    if all(criterios.values()):
        decision = "AMPLIAR"
    elif beneficio_medio <= 0:
        decision = "NO AMPLIAR"
    else:
        decision = "NUEVO EXPERIMENTO"
    no_cumple = [c for c, ok in criterios.items() if not ok]
    return decision, criterios, no_cumple

decision_base, criterios_base, no_cumple_base = motor_decision(conv, beneficio_medio, riesgo)
display(pd.Series(criterios_base).to_frame("cumple"))
print("DECISIÓN DEL MOTOR:", decision_base)
print("Criterios no cumplidos:", no_cumple_base or "ninguno")
print("\nEl motor aplica una política académica fijada antes del análisis; no reemplaza el juicio gerencial.")

,cumple
Efecto positivo con p < 0.05,True
IC 95 % por encima de cero,True
Efecto observado ≥ MDE,True
Beneficio medio simulado > 0,True
Riesgo ≤ tolerancia,True


DECISIÓN DEL MOTOR: AMPLIAR
Criterios no cumplidos: ninguno

El motor aplica una política académica fijada antes del análisis; no reemplaza el juicio gerencial.


**Pregunta 6.1.** Construya una regla reproducible. Para ampliar, exija efecto positivo con p < 0.05, IC por encima de cero, efecto observado ≥ MDE, beneficio medio simulado > 0 y riesgo ≤ tolerancia. Si el beneficio medio es ≤ 0, recomiende no ampliar; en los demás casos, un nuevo experimento.

**Respuesta.** La regla se programó tal como la fija la guía y se aplicó sin cambios. En el caso base se cumplen los cinco criterios: efecto positivo con p = 1.47 × 10⁻¹³, IC [0.500, 0.861] pp por encima de cero, efecto 0.681 pp ≥ MDE 0.5 pp, beneficio medio USD 2 565.64 > 0 y riesgo 0.02 % ≤ 10 %. **Decisión del motor: AMPLIAR.**

In [29]:
# 3 costos × 3 tolerancias con MDE 0.5 pp. Se reutilizan las mismas sumas
# simuladas, así que lo único que cambia entre filas es la política.
filas = []
for costo in COSTOS:
    resultado_c = resultado_incremental(suma_B, suma_A, costo=costo)
    for tol in TOLERANCIAS:
        decision, _, no_cumple = motor_decision(conv, resultado_c.mean(),
                                                np.mean(resultado_c < 0), MDE_BASE, tol)
        filas.append({"costo_USD": costo, "tolerancia": tol,
                      "beneficio_medio": round(resultado_c.mean(), 2),
                      "riesgo": round(np.mean(resultado_c < 0), 4),
                      "P5": round(np.percentile(resultado_c, 5), 2),
                      "decision": decision,
                      "no_cumple": ", ".join(no_cumple) or "—"})
sensibilidad = pd.DataFrame(filas)
display(sensibilidad)
display(sensibilidad.pivot(index="costo_USD", columns="tolerancia", values="decision"))

,costo_USD,tolerancia,beneficio_medio,riesgo,P5,decision,no_cumple
0,0.02,0.05,2865.64,0.0000,1459.97,AMPLIAR,—
1,0.02,0.10,2865.64,0.0000,1459.97,AMPLIAR,—
2,0.02,0.20,2865.64,0.0000,1459.97,AMPLIAR,—
3,0.05,0.05,2565.64,0.0002,1159.97,AMPLIAR,—
4,0.05,0.10,2565.64,0.0002,1159.97,AMPLIAR,—
5,0.05,0.20,2565.64,0.0002,1159.97,AMPLIAR,—
6,0.10,0.05,2065.64,0.0056,659.97,AMPLIAR,—
7,0.10,0.10,2065.64,0.0056,659.97,AMPLIAR,—
8,0.10,0.20,2065.64,0.0056,659.97,AMPLIAR,—


tolerancia,0.05,0.10,0.20
costo_USD,,,
0.02,AMPLIAR,AMPLIAR,AMPLIAR
0.05,AMPLIAR,AMPLIAR,AMPLIAR
0.10,AMPLIAR,AMPLIAR,AMPLIAR


In [30]:
# Caso base (USD 0.05 y 10 %) repetido con MDE de 0.3, 0.5 y 0.8 pp.
filas_mde = []
for mde in MDES:
    decision, _, no_cumple = motor_decision(conv, beneficio_medio, riesgo, mde, TOLERANCIA_BASE)
    filas_mde.append({"MDE_pp": mde * 100, "efecto_observado_pp": round(conv["d"] * 100, 3),
                      "decision": decision, "no_cumple": ", ".join(no_cumple) or "—"})
display(pd.DataFrame(filas_mde))

,MDE_pp,efecto_observado_pp,decision,no_cumple
0,0.3,0.681,AMPLIAR,—
1,0.5,0.681,AMPLIAR,—
2,0.8,0.681,NUEVO EXPERIMENTO,Efecto observado ≥ MDE


In [31]:
# Ingreso incremental por destinatario en cada escenario. Su media es el costo
# desde el cual el beneficio medio deja de ser positivo, y sus percentiles 5, 10
# y 20 son el costo máximo que mantiene el riesgo dentro de cada tolerancia.
ingreso_incremental_cliente = MARGEN * (suma_B - suma_A) / N_FUTURO
umbrales_cambio = pd.DataFrame({
    "condicion": ["Costo que vuelve el beneficio medio ≤ 0 (NO AMPLIAR)"]
                 + [f"Costo máximo con riesgo ≤ {t:.0%}" for t in TOLERANCIAS],
    "costo_por_destinatario_USD": [ingreso_incremental_cliente.mean()]
                 + [np.percentile(ingreso_incremental_cliente, t * 100) for t in TOLERANCIAS]
}).round(4)
display(umbrales_cambio)

,condicion,costo_por_destinatario_USD
0,Costo que vuelve el beneficio medio ≤ 0 (NO AM...,0.3066
1,Costo máximo con riesgo ≤ 5%,0.1660
2,Costo máximo con riesgo ≤ 10%,0.1987
3,Costo máximo con riesgo ≤ 20%,0.2342


**Pregunta 6.2.** Evalúe las nueve combinaciones de los tres costos y las tres tolerancias, con MDE de 0.5 pp. Reutilice los escenarios de gasto para aislar el cambio de política. ¿Cuándo cambia la decisión? Repita el caso base con MDE de 0.3 y 0.8 pp.

**Respuesta.** En las nueve combinaciones de costo (USD 0.02, 0.05, 0.10) y tolerancia (5 %, 10 %, 20 %) la decisión se mantiene en **AMPLIAR**; el peor caso (USD 0.10) deja un beneficio medio de USD 2 065.64, riesgo de 0.56 % y P5 de USD 659.97. Con los mismos escenarios, la decisión cambiaría a **nuevo experimento** si el costo pasa de USD 0.1987 (tolerancia 10 %), de USD 0.1660 (5 %) o de USD 0.2342 (20 %), y a **no ampliar** desde USD 0.3066. Con MDE de 0.3 pp se mantiene AMPLIAR; con **0.8 pp pasa a NUEVO EXPERIMENTO**, porque el efecto observado (0.681 pp) no alcanza ese MDE.

**Pregunta 6.3.** ¿Puede un efecto significativo conducir a no ampliar? ¿Qué condición exacta cambia su recomendación? Explique cómo la ausencia de guardrails observados limita un despliegue real.

**Respuesta.** Sí. Un efecto significativo puede llevar a no ampliar si la campaña no paga su costo: con este mismo efecto, desde USD 0.31 por destinatario el beneficio medio sería ≤ 0. Nuestra recomendación cambia exactamente si el costo supera USD 0.1987 (riesgo > 10 %), si la gerencia exige un MDE mayor a 0.681 pp o si el beneficio medio cae a cero. Como no hay guardrails observados (bajas, quejas), no sabemos si el correo daña la lista de clientes. Un despliegue real debería medirlos en un piloto con umbrales fijados antes.

---
## Tablero de evidencia para la recomendación

Juntamos en una tabla las cifras que usa el informe. Todas salen de variables calculadas arriba; ninguna se escribe a mano.

In [32]:
evidencia = pd.DataFrame({
    "cifra": ["Conversión A", "Conversión B", "Efecto B − A (pp)", "IC 95 % (pp)",
              "p-valor bilateral", "Beneficio esperado puntual (USD)",
              "Intervalo bootstrap 95 % (USD)", "Beneficio medio simulado (USD)",
              "Riesgo modelado", "P5 (USD)", "Decisión caso base"],
    "valor": [f"{p_A:.3%}", f"{p_B:.3%}", f"{conv['d']*100:.3f}",
              f"[{conv['ic95_low']*100:.3f}, {conv['ic95_high']*100:.3f}]",
              f"{conv['p_bilateral']:.3g}", f"{beneficio_puntual:,.2f}",
              f"[{p025:,.2f}, {p975:,.2f}]", f"{beneficio_medio:,.2f}",
              f"{riesgo:.2%}", f"{p5:,.2f}", decision_base]
})
display(evidencia)

,cifra,valor
0,Conversión A,0.573%
1,Conversión B,1.253%
2,Efecto B − A (pp),0.681
3,IC 95 % (pp),"[0.500, 0.861]"
4,p-valor bilateral,1.47e-13
5,Beneficio esperado puntual (USD),"2,579.31"
6,Intervalo bootstrap 95 % (USD),"[1,454.65, 3,731.91]"
7,Beneficio medio simulado (USD),"2,565.64"
8,Riesgo modelado,0.02%
9,P5 (USD),"1,159.97"


---

## **Preguntas escritas**

---


**A. ¿En qué se diferencian el IC del efecto A/B, el intervalo bootstrap del beneficio esperado y los percentiles de la campaña futura?**

El **IC del efecto A/B** [0.500, 0.861] pp describe la incertidumbre sobre un parámetro: la diferencia verdadera de conversión pB − pA en la población del experimento.
>
El **intervalo bootstrap** [USD 1 454.65, USD 3 731.91] describe la incertidumbre de estimación del beneficio **esperado** (promedio) para 10 000 clientes.
>
Los **percentiles del Monte Carlo** (P5 = USD 1 159.97; P95 = USD 3 936.97) describen el rango de resultados que podría **realizar** una campaña futura concreta, bajo los supuestos del modelo.


---

**B. ¿Qué parte del trabajo usa observaciones reales, cuál usa supuestos y cuál produce simulaciones?**

- **Observaciones reales:** las columnas `segment`, `conversion` y `spend` del archivo Hillstrom; por ejemplo, 122 compradores de 21 306 en A y 267 de 21 307 en B.
- **Supuestos:** margen de 40 %, costo de USD 0.05 por destinatario, N = 10 000, MDE de 0.5 pp y tolerancia de 10 %; por ejemplo, el beneficio de USD 2 579.31 depende del margen supuesto.
- **Simulaciones:** las 5 000 réplicas bootstrap y los 5 000 escenarios Monte Carlo; por ejemplo, el riesgo de 0.02 % y el P5 de USD 1 159.97.


---


**C. ¿Qué podría concluirse equivocadamente mirando solo el p-valor o el beneficio promedio?**

Con solo el p-valor (1.47 × 10⁻¹³) se podría pensar que el efecto es grande o seguro. En realidad es de 0.681 pp, su límite inferior apenas toca el MDE (0.500 pp) y el p-valor no dice nada sobre dinero.

>
 Con solo el beneficio promedio (USD 2 565.64) se ocultaría la dispersión: el P5 es USD 1 159.97 y, con costo de USD 0.10, el P5 cae a USD 659.97.
 >
 Tampoco se vería que la decisión se vuelve frágil desde USD 0.20 por destinatario. Y si se promediara solo a los compradores, se concluiría que la campaña reduce el gasto (−USD 0.48).

---

## Informe de recomendación para la gerencia

**Recomendación: AMPLIAR la campaña Mens E-Mail a 10 000 clientes, con seguimiento.**

En el experimento, el correo elevó la conversión de 0.573 % a 1.253 %: **+0.681 pp** (IC 95 %: 0.500 a 0.861 pp; p = 1.5 × 10⁻¹³).
>
 Con margen de 40 % y costo de USD 0.05 por destinatario, el **beneficio medio simulado es USD 2 565.64**, solo el **0.02 %** de los 5 000 escenarios pierde dinero y el **P5 es USD 1 159.97**, dentro de la tolerancia de 10 %.

**Principal riesgo:**
>el límite inferior del efecto apenas alcanza el MDE de 0.5 pp; si la mejora real fuera menor, el beneficio se reduciría.

**Condición de cambio:**
>si el costo por destinatario supera USD 0.20, el riesgo pasa de 10 % y correspondería un nuevo experimento; desde USD 0.31 no convendría ampliar.

**Limitaciones:**
>(1) no se midieron bajas de suscripción ni quejas, así que el despliegue debe monitorear esos guardrails; (2) los datos son de 2008 y el modelo no incluye fatiga publicitaria, cambios de mercado ni dependencia entre clientes.


## **Declaración de uso de inteligencia artificial**


Se utilizó Claude.ia como apoyo para entender los códigos y resolver los ejercicios.
 Y de  esta forma ayudando a entender y comprender cada línea  de codigo antes de incorporarla al notebook.

.

.

.

.

.

.

.

.

.

.

.

.

..

.

.

.

.

.

.

.